In [1]:
!pip install gradio
!pip install ta
!pip install gradio_datetimerange

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os
import gradio as gr
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.losses import MeanSquaredError # Untuk custom_objects
from ta.trend import SMAIndicator, EMAIndicator, MACD
from ta.momentum import RSIIndicator
from ta.volatility import BollingerBands, AverageTrueRange

# --- KONSTANTA PREDIKSI REKURSIF ---
# Ini harus SAMA dengan SEQ_LENGTH yang digunakan saat melatih model
SEQ_LENGTH = 9   # Panjang urutan input historis (harus sama dengan training)

# Maksimum hari yang dapat diprediksi secara rekursif dari data historis terakhir
# Sesuai permintaan: "jika memilih di atas 30 hari itu prediksi tidak bisa"
MAX_FORECAST_DAYS = 9

# --- Konfigurasi Jalur File ---
DATASET_PATH = '/content/Data Historis BBRI (5).csv'
BASE_DIR = os.getcwd()

MODEL_FILENAME = '/content/stock_prediction_model (4).h5'
SCALER_FILENAME = '/content/minmax_scaler (4).pkl'

MODEL_FULL_PATH = os.path.join(BASE_DIR, MODEL_FILENAME)
SCALER_FULL_PATH = os.path.join(BASE_DIR, SCALER_FILENAME)

# --- Memuat Model dan Scaler yang Sudah Terlatih ---
try:
    # Memuat model dengan custom_objects untuk fungsi loss
    model = load_model(MODEL_FULL_PATH, custom_objects={'mse': MeanSquaredError()})
    scaler = joblib.load(SCALER_FULL_PATH)
    print(f"Model '{MODEL_FILENAME}' dan Scaler '{SCALER_FILENAME}' berhasil dimuat.")
except Exception as e:
    print(f"Gagal memuat model atau scaler: {e}")
    print("Pastikan Anda sudah menjalankan 'train_model.py' terlebih dahulu untuk menghasilkan file tersebut dan pastikan file ada di lokasi yang benar.")
    print(f"Mencari model di: {MODEL_FULL_PATH}")
    print(f"Mencari scaler di: {SCALER_FULL_PATH}")
    exit()

# --- Fungsi Pembersihan dan Pra-pemrosesan Data ---
def clean_volume(vol):
    """Membersihkan dan mengkonversi kolom 'Vol.' dari string ke float."""
    vol = str(vol).replace(',', '.')
    if 'M' in vol:
        return float(vol.replace('M', '')) * 1e6
    elif 'B' in vol:
        return float(vol.replace('B', '')) * 1e9
    else:
        return np.nan

# Memuat dan pra-pemrosesan dataset global
print("=== Memuat Dataset untuk Antarmuka Gradio ===")
try:
    df_global = pd.read_csv(DATASET_PATH)
    # Normalisasi Tanggal di df_global ke tengah malam (00:00:00)
    df_global['Tanggal'] = pd.to_datetime(df_global['Tanggal'], format='%d/%m/%Y').dt.normalize()
    df_global = df_global.sort_values('Tanggal').reset_index(drop=True)

    df_global['Vol.'] = df_global['Vol.'].apply(clean_volume)
    df_global['Perubahan%'] = df_global['Perubahan%'].str.replace('%', '').str.replace(',', '.').astype(float)
    for col in ['Terakhir', 'Pembukaan', 'Tertinggi', 'Terendah']:
        df_global[col] = df_global[col].astype(float)
    print("Dataset berhasil dimuat dan diproses untuk Gradio.")
    print(f"Jumlah baris data global: {len(df_global)}")
except Exception as e:
    print(f"Gagal memuat atau memproses dataset global: {e}")
    df_global = pd.DataFrame()

# --- Fungsi Perhitungan Indikator Teknis ---
def calculate_technical_indicators(df_input):
    """Menghitung indikator teknis. Harus sama dengan train_model.py."""
    df = df_input.copy()
    df['SMA_5'] = SMAIndicator(df['Terakhir'], window=5).sma_indicator()
    df['EMA_5'] = EMAIndicator(df['Terakhir'], window=5).ema_indicator()
    df['RSI_14'] = RSIIndicator(df['Terakhir'], window=14).rsi()
    macd_instance = MACD(df['Terakhir'])
    df['MACD'] = macd_instance.macd()
    df['Signal_Line'] = macd_instance.macd_signal()

    bollinger = BollingerBands(close=df['Terakhir'], window=20, window_dev=2)
    df['BB_Mid'] = bollinger.bollinger_mavg()
    df['BB_High'] = bollinger.bollinger_hband()
    df['BB_Low'] = bollinger.bollinger_lband()
    df['BB_Width'] = bollinger.bollinger_wband()
    df['BB_Percent'] = bollinger.bollinger_pband()

    atr = AverageTrueRange(high=df['Tertinggi'], low=df['Terendah'], close=df['Terakhir'], window=14)
    df['ATR'] = atr.average_true_range()
    return df

# --- Fungsi Prediksi Utama untuk Gradio (dengan Prediksi Rekursif) ---
def predict_stock_price(tanggal_target_prediksi):
    """
    Fungsi ini dipanggil oleh antarmuka Gradio untuk memprediksi harga saham
    untuk tanggal target spesifik menggunakan prediksi rekursif.
    """
    print(f"\n=== Fungsi Prediksi Dipanggil ===")
    print(f"Tanggal Target Prediksi yang dipilih: {tanggal_target_prediksi}")

    if df_global.empty:
        return "Dataset tidak dimuat. Harap periksa file data Anda.", None

    if tanggal_target_prediksi is None:
        return "Pilih tanggal target prediksi terlebih dahulu.", None

    try:
        # Normalisasi tanggal input dari Gradio ke tengah malam (00:00:00)
        tanggal_target_prediksi_dt = pd.to_datetime(tanggal_target_prediksi).normalize()
    except Exception as e:
        print(f"Error parsing date: {e}")
        return "Format tanggal salah! Gunakan kalender untuk memilih tanggal.", None

    # Normalisasi tanggal historis terakhir ke tengah malam
    last_historical_date = df_global['Tanggal'].max().normalize()

    # Validasi Tanggal Target Prediksi
    if tanggal_target_prediksi_dt <= last_historical_date:
        return f"Tanggal target prediksi ({tanggal_target_prediksi_dt.strftime('%Y-%m-%d')}) " \
               f"harus setelah tanggal historis terakhir ({last_historical_date.strftime('%Y-%m-%d')}).", None

    # Perhitungan langkah rekursif menggunakan tanggal yang sudah dinormalisasi
    num_recursive_steps = (tanggal_target_prediksi_dt - last_historical_date).days

    if num_recursive_steps <= 0:
        return "Tanggal target prediksi harus di masa depan dari data historis.", None

    if num_recursive_steps > MAX_FORECAST_DAYS:
        return f"Prediksi tidak dapat dilakukan lebih dari {MAX_FORECAST_DAYS} hari. Tanggal target Anda terlalu jauh ({num_recursive_steps} hari).", None

    # Tentukan data historis yang akan digunakan sebagai basis
    # Kita selalu memprediksi dari akhir data historis yang tersedia
    df_base = df_global.copy()
    actual_base_date_for_prediction = last_historical_date # Sudah dinormalisasi
    print(f"Menggunakan data historis hingga: {actual_base_date_for_prediction.strftime('%Y-%m-%d')} sebagai dasar prediksi.")

    # Hitung indikator teknis pada data dasar
    df_with_indicators = calculate_technical_indicators(df_base)

    features_to_scale = ['Terakhir', 'Pembukaan', 'Tertinggi', 'Terendah', 'Vol.', 'Perubahan%',
                         'SMA_5', 'EMA_5', 'RSI_14', 'MACD', 'Signal_Line',
                         'BB_Mid', 'BB_High', 'BB_Low', 'BB_Width', 'BB_Percent', 'ATR']
    df_features = df_with_indicators[features_to_scale].dropna()

    if len(df_features) < SEQ_LENGTH:
        return f"Data historis yang tersedia ({len(df_features)} hari) tidak cukup untuk membuat sequence input ({SEQ_LENGTH} hari).", None

    try:
        # --- LOGIKA PREDIKSI REKURSIF ---
        current_sequence_raw = df_features.tail(SEQ_LENGTH).values
        current_sequence_scaled_list = scaler.transform(current_sequence_raw).tolist()

        predicted_prices_for_path = [] # Untuk menyimpan semua prediksi langkah demi langkah
        predicted_dates_for_path = []

        # Tanggal awal untuk prediksi (hari setelah tanggal dasar prediksi aktual)
        current_prediction_date = actual_base_date_for_prediction + pd.Timedelta(days=1)

        final_predicted_price = None # Akan menyimpan prediksi untuk tanggal target terakhir

        for i in range(num_recursive_steps):
            reshaped_sequence = np.array(current_sequence_scaled_list).reshape(1, SEQ_LENGTH, len(features_to_scale))
            predicted_scaled_price = model.predict(reshaped_sequence, verbose=0)[0][0]

            dummy_row_for_inverse = np.zeros((1, len(features_to_scale)))
            dummy_row_for_inverse[0, 0] = predicted_scaled_price
            predicted_unscaled_price = scaler.inverse_transform(dummy_row_for_inverse)[0, 0]

            predicted_prices_for_path.append(predicted_unscaled_price)
            predicted_dates_for_path.append(current_prediction_date)

            # Jika ini adalah prediksi untuk tanggal target, simpan harganya
            # Pastikan perbandingan tanggal dilakukan dengan normalize() untuk mengabaikan waktu
            if current_prediction_date.normalize() == tanggal_target_prediksi_dt.normalize():
                final_predicted_price = predicted_unscaled_price

            # Perbarui current_sequence untuk prediksi langkah berikutnya
            last_scaled_input_features = current_sequence_scaled_list[-1]
            new_scaled_feature_row = list(last_scaled_input_features)
            new_scaled_feature_row[0] = predicted_scaled_price

            current_sequence_scaled_list.pop(0)
            current_sequence_scaled_list.append(new_scaled_feature_row)

            current_prediction_date += pd.Timedelta(days=1)

        # --- Hasil Prediksi Teks ---
        if final_predicted_price is None:
             result_text = "Terjadi kesalahan: Harga target prediksi tidak ditemukan. (Mungkin ada masalah dengan tanggal atau loop)"
        else:
             result_text = f"Prediksi harga saham BBRI untuk tanggal {tanggal_target_prediksi_dt.strftime('%Y-%m-%d')}: Rp {final_predicted_price:.2f}"
             if num_recursive_steps > 1:
                result_text += f"\n(Dihitung secara rekursif dalam {num_recursive_steps} langkah dari {actual_base_date_for_prediction.strftime('%Y-%m-%d')})"

        # --- Plotting Hasil ---
        fig_comparison = plt.figure(figsize=(14, 7))

        plt.plot(df_base['Tanggal'], df_base['Terakhir'], label='Harga Asli Historis', color='blue')

        # Hanya plot satu titik prediksi untuk tanggal target
        if final_predicted_price is not None:
            plt.plot([tanggal_target_prediksi_dt], [final_predicted_price], 'ro', markersize=8, label=f'Prediksi {tanggal_target_prediksi_dt.strftime("%Y-%m-%d")}')

            # Opsional: Jika Anda ingin melihat 'jalur' prediksi rekursif ke tanggal target, aktifkan baris ini:
            # plt.plot(predicted_dates_for_path, predicted_prices_for_path, label=f'Jalur Prediksi Rekursif', color='red', linestyle=':', alpha=0.6)

        plt.axvline(x=actual_base_date_for_prediction, color='gray', linestyle=':', label='Tanggal Historis Terakhir')
        plt.axvline(x=tanggal_target_prediksi_dt, color='green', linestyle='--', label='Tanggal Target Prediksi')

        plt.title(f'Harga Saham BBRI Historis dan Prediksi untuk {tanggal_target_prediksi_dt.strftime("%Y-%m-%d")}')
        plt.xlabel('Tanggal')
        plt.ylabel('Harga (Rp)')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()

        return result_text, fig_comparison

    except Exception as e:
        print(f"Terjadi kesalahan tak terduga saat memprediksi: {str(e)}")
        return f"Terjadi kesalahan saat memprediksi: {str(e)}", None

# --- Menyiapkan Antarmuka Gradio ---
print("\n=== Menyiapkan Antarmuka Gradio ===")
with gr.Blocks(title="Prediksi Harga Saham BBRI") as demo:
    gr.Markdown("# Prediksi Harga Saham BBRI")
    gr.Markdown(f"Pilih tanggal spesifik di masa depan untuk mendapatkan prediksi harga saham BBRI.")
    gr.Markdown(f"Prediksi dapat dilakukan hingga maksimal **{MAX_FORECAST_DAYS} hari** dari tanggal historis terakhir Anda ({df_global['Tanggal'].max().strftime('%Y-%m-%d')}).")

    with gr.Row():
        # gr.DateTime untuk memilih tanggal target prediksi
        # Normalisasi nilai default ke tengah malam agar konsisten
        date_input = gr.DateTime(label="Tanggal Target Prediksi (Forecasting Window)", type="datetime",
                                 value=(pd.Timestamp.now().normalize() + pd.Timedelta(days=1)).strftime('%Y-%m-%d %H:00:00'),
                                 info=f"Pilih tanggal di masa depan (maksimal {MAX_FORECAST_DAYS} hari dari data historis terakhir).")

        predict_btn = gr.Button("Prediksi Harga")

    output_text = gr.Textbox(label="Hasil Prediksi", interactive=False, lines=5)
    output_plot = gr.Plot(label="Grafik Historis dan Prediksi")

    predict_btn.click(
        fn=predict_stock_price,
        inputs=[date_input],
        outputs=[output_text, output_plot]
    )

print("\nMeluncurkan antarmuka Gradio. Buka URL yang ditampilkan di browser Anda.")
demo.launch(debug=True)


Model '/content/stock_prediction_model (4).h5' dan Scaler '/content/minmax_scaler (4).pkl' berhasil dimuat.
=== Memuat Dataset untuk Antarmuka Gradio ===
Dataset berhasil dimuat dan diproses untuk Gradio.
Jumlah baris data global: 826

=== Menyiapkan Antarmuka Gradio ===

Meluncurkan antarmuka Gradio. Buka URL yang ditampilkan di browser Anda.
Setting queue=True in a Colab notebook requires sharing enabled. Setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Running on public URL: https://0fbb075bcac1e76d6e.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)



=== Fungsi Prediksi Dipanggil ===
Tanggal Target Prediksi yang dipilih: 2025-06-23 00:00:00
Menggunakan data historis hingga: 2025-06-20 sebagai dasar prediksi.


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(



=== Fungsi Prediksi Dipanggil ===
Tanggal Target Prediksi yang dipilih: 2025-06-21 00:00:00
Menggunakan data historis hingga: 2025-06-20 sebagai dasar prediksi.


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


...


=== Fungsi Prediksi Dipanggil ===
Tanggal Target Prediksi yang dipilih: 2025-06-25 00:00:00
Menggunakan data historis hingga: 2025-06-20 sebagai dasar prediksi.


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(



=== Fungsi Prediksi Dipanggil ===
Tanggal Target Prediksi yang dipilih: 2025-06-26 00:00:00
Menggunakan data historis hingga: 2025-06-20 sebagai dasar prediksi.


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(



=== Fungsi Prediksi Dipanggil ===
Tanggal Target Prediksi yang dipilih: 2025-06-30 00:00:00

=== Fungsi Prediksi Dipanggil ===
Tanggal Target Prediksi yang dipilih: 2025-06-19 00:00:00
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://0fbb075bcac1e76d6e.gradio.live
